# NC-Musical: AI Music Transcription & Interactive Editor
Run GPU-accelerated automatic music transcription (AMT) using the MuScriptor engine and edit transcriptions interactively in your web browser via the Piano Roll Web GUI.

**Note:** This notebook mounts Google Drive to permanently store model weights (`~1.2GB`) and SoundFont files so you don't have to re-download them every session.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentHitmanFaris/NC-Musical/blob/Stable/NC_Musical_Colab.ipynb)

In [ ]:
# @title 1. Mount Google Drive (Persistent Model Storage)
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

# Directory in Google Drive for caching models permanently
drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
os.makedirs(drive_models_dir, exist_ok=True)
hf_cache_dir = os.path.join(drive_models_dir, "huggingface")
os.makedirs(hf_cache_dir, exist_ok=True)

# Set HuggingFace and PyTorch cache dirs to Google Drive
os.environ["HF_HOME"] = hf_cache_dir
os.environ["TORCH_HOME"] = os.path.join(drive_models_dir, "torch")

print(f"All AI models will be saved permanently to: {drive_models_dir}")


In [ ]:
# @title 2. Check GPU & Setup Repository
import os
import sys
import subprocess
import getpass

print("Checking GPU environment...")
!nvidia-smi

print("\nInstalling system dependencies (FFmpeg and FluidSynth)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fluidsynth

print("\nCloning NC-Musical repository...")
repo_dir = "/content/NC-Musical"
if not os.path.exists(repo_dir):
    res = subprocess.run(["git", "clone", "https://github.com/AgentHitmanFaris/NC-Musical.git", repo_dir], capture_output=True, text=True)
    if res.returncode != 0:
        print("Public clone failed (Repository is private). Access Token required.")
        token = None
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
        
        if not token:
            print("\nPlease enter your GitHub Personal Access Token (PAT) to clone the private repository:")
            token = getpass.getpass("GitHub Token: ")
        
        token = token.strip()
        !git clone https://{token}@github.com/AgentHitmanFaris/NC-Musical.git /content/NC-Musical

if os.path.exists(repo_dir):
    %cd /content/NC-Musical
    print("\nInstalling Python dependencies...")
    !pip install -q uvicorn fastapi yt-dlp soundfile torch torchvision torchaudio
    !pip install -q muscriptor || true

    print("\nRunning patch script...")
    !python patch_muscriptor.py || true
else:
    print("Error: Could not clone repository.")


In [ ]:
# @title 3. Setup MS Basic.sf3 SoundFont in Google Drive
import os
import urllib.request

drive_models_dir = "/content/drive/MyDrive/NC-Musical-Models"
drive_sf3_path = os.path.join(drive_models_dir, "MS Basic.sf3")
local_sf3_path = "/content/NC-Musical/MS Basic.sf3"

if not os.path.exists(drive_sf3_path):
    print("Downloading MS Basic.sf3 SoundFont (~50MB) to Google Drive...")
    sf3_urls = [
        "https://raw.githubusercontent.com/musescore/MuseScore/v4.1.0/share/sound/MS%20Basic.sf3",
        "https://huggingface.co/MuScriptor/assets/resolve/main/MuseScore_General.sf3"
    ]
    downloaded = False
    for url in sf3_urls:
        try:
            urllib.request.urlretrieve(url, drive_sf3_path)
            print("MS Basic.sf3 downloaded and saved to Google Drive!")
            downloaded = True
            break
        except Exception as e:
            print(f"Notice ({e}): trying next source...")
    if not downloaded:
        print("Warning: SoundFont download failed.")
else:
    print("MS Basic.sf3 SoundFont found in Google Drive!")

if os.path.exists(drive_sf3_path) and not os.path.exists(local_sf3_path):
    !cp "$drive_sf3_path" "$local_sf3_path"


In [ ]:
# @title 4. Start Server & Open Web GUI
import os
import time
import subprocess
from google.colab import output

# Kill any existing server instance
!pkill -f "server_gui.py" || true

PORT = 8222

print("Starting MuScriptor FastAPI server on GPU...")
env = os.environ.copy()

server_process = subprocess.Popen(
    ["python", "server_gui.py", "--port", str(PORT), "--model", "large", "--device", "cuda"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env
)

time.sleep(3)

# Primary Colab Proxy Link
try:
    colab_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    if not colab_url.endswith("/"):
        colab_url += "/"
    print("\n" + "="*65)
    print("SUCCESS! Click the link below to open the MuScriptor Web GUI:")
    print(f"--> {colab_url}index.html <--")
    print("="*65 + "\n")
except Exception as e:
    print("Colab proxy link notice:", e)

# Alternative Cloudflare Tunnel Link
print("Creating alternative Cloudflare Tunnel link...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(4)
for _ in range(20):
    line = cf_process.stdout.readline()
    if "trycloudflare.com" in line:
        url = line.strip().split()[-1]
        if not url.endswith("/"):
            url += "/"
        print(f"Alternative Public Link (Cloudflare): {url}index.html")
        break
    time.sleep(0.5)
